# Train and validate NicheTrans with schema-compliant H5MU files

This notebook consumes two H5MU subsets that were prepared in advance: one for training and one for periodic validation/model selection. It does not split either file. Edit the paths and settings in the configuration cell before running.

In [ ]:
from pathlib import Path
import warnings

import pandas as pd
import torch
import torch.nn as nn
from IPython.display import display
from torch.optim import lr_scheduler

from args.args_h5mu import generate_args
from datasets.h5mu_dataset import H5MuDataManager, validate_h5mu
from model.nicheTrans import NicheTrans
from utils.utils import set_seed
from utils.utils_h5mu_dataloader import h5mu_dataloader
from utils.utils_training_h5mu import (
    build_criterion,
    evaluate,
    fit,
    infer_task_type,
    resolve_device,
)

## Configuration

Replace both placeholder paths. `task=auto` maps target `value_type=binary` to binary classification and all other value types to regression.

In [ ]:
TRAIN_H5MU = Path(r"D:\\path\\to\\train_subset.h5mu")
TEST_H5MU = Path(r"D:\\path\\to\\test_subset.h5mu")

args = generate_args([
    "--train-path", str(TRAIN_H5MU),
    "--test-path", str(TEST_H5MU),
    "--source-modality", "rna",
    "--target-modality", "protein",
    "--n-neighbors", "12",
    "--preprocess", "auto",
    "--task", "auto",
    "--max-epoch", "40",
    "--eval-step", "1",
    "--train-batch", "32",
    "--test-batch", "32",
    "--workers", "4",
    "--device", "auto",
    "--output-dir", "outputs/h5mu",
    "--run-name", "nichetrans_h5mu",
])
display(vars(args))

## Validate and load both files

Feature names and order must be identical between the two files. Spatial neighbors are built independently inside each file and `sample_id`.

In [ ]:
for path, label in [(Path(args.train_path), "training"), (Path(args.test_path), "testing")]:
    if not path.is_file():
        raise FileNotFoundError(f"Set the {label} H5MU path in the configuration cell: {path}")

train_metadata = validate_h5mu(args.train_path)
test_metadata = validate_h5mu(args.test_path)
dataset = H5MuDataManager(
    train_path=args.train_path,
    test_path=args.test_path,
    source_modality=args.source_modality,
    target_modality=args.target_modality,
    n_neighbors=args.n_neighbors,
    preprocess=args.preprocess,
    rna_target_sum=args.rna_target_sum,
)

train_value_type = dataset.train_assays[args.target_modality]["value_type"]
test_value_type = dataset.test_assays[args.target_modality]["value_type"]
task = infer_task_type(train_value_type, test_value_type, requested=args.task)
trainloader, validationloader = h5mu_dataloader(args, dataset)

print(f"Task: {task}; source: {dataset.source_length}; target: {dataset.target_length}")
print(f"Training observations: {len(dataset.training):,}")
print(f"Validation observations: {len(dataset.testing):,}")

## Model and optimization

This workflow intentionally uses the standard NicheTrans model. Targets wider than 512 features are supported but may be slow because the standard model uses one prediction head per target.

In [ ]:
set_seed(args.seed)
device = resolve_device(args.device)
if dataset.target_length > 512:
    warnings.warn(
        f"The standard NicheTrans model has {dataset.target_length} target heads; "
        "training may be slow or memory intensive."
    )

model = NicheTrans(
    source_length=dataset.source_length,
    target_length=dataset.target_length,
    noise_rate=args.noise_rate,
    dropout_rate=args.dropout_rate,
).to(device)
if device.type == "cuda" and torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)

criterion = build_criterion(task)
if args.optimizer == "adam":
    optimizer = torch.optim.Adam(
        model.parameters(), lr=args.lr, weight_decay=args.weight_decay
    )
else:
    optimizer = torch.optim.SGD(
        model.parameters(), lr=args.lr, weight_decay=args.weight_decay
    )
scheduler = (
    lr_scheduler.StepLR(optimizer, step_size=args.stepsize, gamma=args.gamma)
    if args.stepsize > 0
    else None
)
print(f"Device: {device}; model: {type(model).__name__}; loss: {type(criterion).__name__}")

## Train with periodic validation

Regression runs maximize mean Pearson correlation; binary runs maximize mean AUROC. Validation loss is used when the primary metric cannot be computed.

In [ ]:
output_dir = Path(args.output_dir)
output_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = output_dir / f"{args.run_name}_best.pth"
checkpoint_metadata = {
    "args": vars(args),
    "source_panel": [str(value) for value in dataset.source_panel],
    "target_panel": [str(value) for value in dataset.target_panel],
    "train_database": dataset.train_database,
    "test_database": dataset.test_database,
    "train_assays": dataset.train_assays,
    "test_assays": dataset.test_assays,
}

result = fit(
    model=model,
    trainloader=trainloader,
    validationloader=validationloader,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    task=task,
    max_epochs=args.max_epoch,
    eval_step=args.eval_step,
    checkpoint_path=checkpoint_path,
    scheduler=scheduler,
    target_panel=dataset.target_panel,
    checkpoint_metadata=checkpoint_metadata,
    neighbor_mask_probability=args.neighbor_mask_probability,
    neighbor_keep_probability=args.neighbor_keep_probability,
)

## Evaluate the restored best model and save reports

In [ ]:
final_evaluation = evaluate(
    model=model,
    dataloader=validationloader,
    criterion=criterion,
    device=device,
    task=task,
    target_panel=dataset.target_panel,
)
history_df = pd.DataFrame(result["history"])
feature_metrics_df = pd.DataFrame(final_evaluation["per_feature"])
history_path = output_dir / f"{args.run_name}_history.csv"
metrics_path = output_dir / f"{args.run_name}_validation_metrics.csv"
history_df.to_csv(history_path, index=False)
feature_metrics_df.to_csv(metrics_path, index=False)

print(f"Best epoch: {result['best_epoch']}")
print(f"Checkpoint: {result['checkpoint_path']}")
print(f"History: {history_path}")
print(f"Per-feature metrics: {metrics_path}")
display(pd.DataFrame([final_evaluation["summary"]]))
display(feature_metrics_df.head(20))

## Close backed H5MU handles

Run this cell when training finishes or before changing input files.

In [ ]:
dataset.close()
print("H5MU file handles closed.")